# YOLO Fruit Detection Training

Train YOLO v8 model for multi-class fruit detection (Mango, Grapefruit, Guava, Orange)

In [ ]:
# Install Required Packages (Run this first in Google Colab)
!pip install ultralytics
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("Packages installed successfully!")

In [ ]:
# Mount Google Drive (For Google Colab)
try:
    from google.colab import drive
    import os
    
    # Mount Google Drive
    drive.mount('/content/drive')
    
    # Change to your dataset directory (adjust path as needed)
    # The dataset is directly in Google Drive MyDrive as multi_fruit_detection
    dataset_path = '/content/drive/MyDrive/multi_fruit_detection'
    if os.path.exists(dataset_path):
        print(f"✅ Found dataset directory: {dataset_path}")
        print("Dataset contents:")
        print([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])
    else:
        print(f"❌ Dataset directory not found: {dataset_path}")
        print("Please upload your multi_fruit_detection folder to Google Drive MyDrive")
        print("Available directories in MyDrive:")
        mydrive_path = '/content/drive/MyDrive'
        if os.path.exists(mydrive_path):
            print([d for d in os.listdir(mydrive_path) if os.path.isdir(os.path.join(mydrive_path, d))])
    
    print("Google Drive mounted successfully!")
    
except ImportError:
    print("Not running in Google Colab, skipping drive mount...")
    print("Using local dataset path...")

In [ ]:
# Import Required Libraries
import os
import torch
from ultralytics import YOLO
import yaml
from pathlib import Path

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Verify Dataset Structure
from pathlib import Path

# Check if we're in Google Colab or local environment
try:
    from google.colab import drive
    dataset_path = Path('/content/drive/MyDrive/multi_fruit_detection')
    print("🔍 Running in Google Colab")
except ImportError:
    dataset_path = Path('data/unified/multi_fruit_detection')
    print("🔍 Running locally")

print(f"Dataset path: {dataset_path}")

# Quick check for required structure
if not dataset_path.exists():
    print("❌ Dataset not found!")
    print("Please upload your multi_fruit_detection folder to Google Drive MyDrive")
else:
    print("✅ Dataset directory found!")
    
    required_dirs = ['train/images', 'train/labels', 'val/images', 'val/labels']
    all_exist = all((dataset_path / d).exists() for d in required_dirs)
    
    if all_exist:
        print("✅ All required directories present!")
    else:
        print("⚠️ Some required directories are missing!")
        for d in required_dirs:
            status = "✅" if (dataset_path / d).exists() else "❌"
            print(f"{status} {d}")

In [ ]:
# Create data.yaml Configuration
yaml_content = {
    'path': str(dataset_path),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 4,
    'names': ['mango', 'grapefruit', 'guava', 'orange']
}

# Save yaml file
yaml_file = dataset_path / 'data.yaml'
with open(yaml_file, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("✅ Created data.yaml configuration")
print(f"Classes: {yaml_content['nc']}")
print(f"Names: {yaml_content['names']}")

In [ ]:
# Initialize YOLO Model
model = YOLO('yolov8n.pt')
print("✅ YOLOv8 Nano model loaded successfully!")

In [ ]:
# Train YOLO Model
results = model.train(
    data=str(yaml_file),
    epochs=100,
    imgsz=640,
    batch=16,
    name='multi_fruit_detection_v1',
    patience=10,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
    plots=True
)

print("🎉 Training completed!")
print("📁 Results saved in: runs/detect/multi_fruit_detection_v1/")

In [ ]:
# Evaluate Model - Comprehensive Results
from IPython.display import Image, display
import os

# Load best model
best_model = YOLO('runs/detect/multi_fruit_detection_v1/weights/best.pt')
print("✅ Best model loaded\n")

# Evaluate on validation set
metrics = best_model.val()

# Display comprehensive performance metrics
print("="*60)
print("📊 MODEL PERFORMANCE METRICS")
print("="*60)
print(f"mAP@0.5:        {metrics.box.map50:.4f} {'✅ Excellent' if metrics.box.map50 > 0.85 else '⚠️ Needs improvement' if metrics.box.map50 > 0.70 else '❌ Poor'}")
print(f"mAP@0.5-0.95:   {metrics.box.map:.4f} {'✅ Excellent' if metrics.box.map > 0.70 else '⚠️ Good' if metrics.box.map > 0.50 else '❌ Needs improvement'}")
print(f"Precision:      {metrics.box.mp:.4f} {'✅ High' if metrics.box.mp > 0.85 else '⚠️ Moderate' if metrics.box.mp > 0.70 else '❌ Low'}")
print(f"Recall:         {metrics.box.mr:.4f} {'✅ High' if metrics.box.mr > 0.85 else '⚠️ Moderate' if metrics.box.mr > 0.70 else '❌ Low'}")
print("="*60)

# Performance assessment
print("\n🎯 TRAINING QUALITY ASSESSMENT:")
if metrics.box.map50 > 0.85 and metrics.box.map > 0.65:
    print("✅ Model is WELL-TRAINED and ready for deployment!")
elif metrics.box.map50 > 0.70:
    print("⚠️ Model shows GOOD performance but could be improved")
else:
    print("❌ Model needs MORE TRAINING or data improvement")

# Display per-class metrics if available
print("\n📋 PER-CLASS PERFORMANCE:")
class_names = ['mango', 'grapefruit', 'guava', 'orange']
if hasattr(metrics.box, 'ap50'):
    for i, name in enumerate(class_names):
        if i < len(metrics.box.ap50):
            ap = metrics.box.ap50[i]
            print(f"{name:12s}: {ap:.4f}")

# Display all available training visualizations
print("\n📈 TRAINING VISUALIZATIONS:")
results_dir = 'runs/detect/multi_fruit_detection_v1'

# 1. Training curves (Loss, mAP, Precision, Recall)
if os.path.exists(f'{results_dir}/results.png'):
    print("\n1️⃣ Training Curves (Loss, Metrics over Epochs):")
    display(Image(f'{results_dir}/results.png'))

# 2. Confusion Matrix
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    print("\n2️⃣ Confusion Matrix:")
    display(Image(f'{results_dir}/confusion_matrix.png'))

# 3. F1 Curve
if os.path.exists(f'{results_dir}/F1_curve.png'):
    print("\n3️⃣ F1-Confidence Curve:")
    display(Image(f'{results_dir}/F1_curve.png'))

# 4. Precision-Recall Curve
if os.path.exists(f'{results_dir}/PR_curve.png'):
    print("\n4️⃣ Precision-Recall Curve:")
    display(Image(f'{results_dir}/PR_curve.png'))

# 5. Precision Curve
if os.path.exists(f'{results_dir}/P_curve.png'):
    print("\n5️⃣ Precision-Confidence Curve:")
    display(Image(f'{results_dir}/P_curve.png'))

# 6. Recall Curve
if os.path.exists(f'{results_dir}/R_curve.png'):
    print("\n6️⃣ Recall-Confidence Curve:")
    display(Image(f'{results_dir}/R_curve.png'))

# 7. Validation Batch Labels
if os.path.exists(f'{results_dir}/val_batch0_labels.jpg'):
    print("\n7️⃣ Validation Batch - Ground Truth Labels:")
    display(Image(f'{results_dir}/val_batch0_labels.jpg'))

# 8. Validation Batch Predictions
if os.path.exists(f'{results_dir}/val_batch0_pred.jpg'):
    print("\n8️⃣ Validation Batch - Model Predictions:")
    display(Image(f'{results_dir}/val_batch0_pred.jpg'))

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE - Review the metrics and visualizations above")
print("="*60)